# Example 25: Interactive Workspace (topoview) Demo - 7D Torus

In this notebook, we showcase the `visualize_topoview` API, which provides a high-level interactive workspace for exploring the topological and geometric invariants of a simplicial complex.

We will:
1. Generate a **250-point 7-dimensional torus** ($T^2$) embedded in $\mathbb{R}^7$.
2. Construct its **simplicial complex** using the CkNN method.
3. **Calculate invariants** ($H_0, H_1, H_2, \pi_1$) beforehand.
4. Use `topoview` to visualize the workspace using these pre-calculated inputs.

In [1]:
import numpy as np
import pysurgery as ps
from pysurgery.topoview import visualize_topoview
from pysurgery.bridge.julia_bridge import julia_engine
from time import time

In [2]:
if julia_engine.available:
    print("Julia bridge is available.")
    julia_engine.warmup()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/home/gabriel/Desktop/SurgeryTheory/pysurgery/bridge/julia_bridge.py:113: UserWarning: Julia warm-up completed with partial failures; regular execution will still proceed. Failed workloads: ['h1_valid_square', 'h2_tetra_boundary']
  self._run_warmup(mode)


Julia bridge is available.
Starting SNF calculation for 2x2 matrix...
Finished SNF calculation.


/home/gabriel/Desktop/SurgeryTheory/pysurgery/bridge/julia_bridge.py:514: UserWarning: Julia warm-up completed with partial failures; regular execution will still proceed. Failed workloads: ['h1_valid_square', 'h2_tetra_boundary']
  return self._run_warmup("full")


In [3]:
# 1. Generate 250 points on a 2-torus in R7
n_points = 250
theta = np.random.uniform(0, 2 * np.pi, n_points)
phi = np.random.uniform(0, 2 * np.pi, n_points)
R, r = 2.0, 1

points_7d_torus = np.column_stack([
    np.zeros(n_points),                    # x1
    np.zeros(n_points),                    # x2
    (R + r * np.cos(theta)) * np.cos(phi), # x3
    (R + r * np.cos(theta)) * np.sin(phi), # x4
    r * np.sin(theta),                     # x5
    np.zeros(n_points),                    # x6
    np.zeros(n_points),                    # x7
])

print(f"Generated {n_points} points in 7D representing a torus.")

Generated 250 points in 7D representing a torus.


In [4]:
# 2. Build the complex
sc = ps.SimplicialComplex.from_point_cloud_cknn(
    points_7d_torus,
    k=21,
    delta=1.5,
    max_dimension=3 # dimension must be one higher than we want to satisfy link condition in simplification
)

# this takes a while
#sc = ps.SimplicialComplex.from_alpha_complex(
#    points_7d_torus,
#    max_alpha_square=2,
#)

#sc = ps.SimplicialComplex.from_vietoris_rips(
#    points_7d_torus,
#    epsilon=2.5,
#    max_dimension=3
#)
sc._coordinates = points_7d_torus

print(f"Simplicial Complex built with {sc.cells_count} simplices.")

Simplicial Complex built with {0: 250, 1: 6825, 2: 75453, 3: 478270} simplices.


In [5]:
# 3. Simplifying the complex to faster calculations
sc_qm, simplex_map = sc.simplify()
print(f"Simplicial Complex reduced from {sc.cells_count} to {sc_qm.cells_count} simplices")

Simplicial Complex reduced from {0: 250, 1: 6825, 2: 75453, 3: 478270} to {0: 250, 1: 6825, 2: 75453, 3: 478270} simplices


## Calculating Invariants on Simplified Complex

In [ ]:
t0 = time()
h0_qm = ps.compute_homology_basis_from_complex(sc_qm, dimension=0, mode='optimal')
h1_qm = ps.compute_homology_basis_from_complex(sc_qm, dimension=1, mode='optimal')
h2_qm = ps.compute_homology_basis_from_complex(sc_qm, dimension=2, mode='optimal')
pi1_qm = ps.extract_pi_1_with_traces(sc_qm.to_cw_complex(), simplify=True, generator_mode='optimized')

print(f"Invariants computed in {time() - t0:.2f} seconds.")
print(f"H0 Rank: {h0_qm.rank}, H1 Rank: {h1_qm.rank}, H2 Rank: {h2_qm.rank}") # must be (1, 2, 1)

## Lifting Invariants to Original Geometry

In [ ]:
from pysurgery.core.generator_models import HomologyBasisResult, HomologyGenerator, Pi1PresentationWithTraces, Pi1GeneratorTrace
import networkx as nx

# Reconstruct original 1-skeleton for path finding
orig_G = nx.Graph()
for e in sc.n_simplices(1):
    orig_G.add_edge(e[0], e[1])

def lift_h_basis(h_basis, dim):
    lifted_gens = []
    for gen in h_basis.generators:
        # The simplex_map gives us ALL original simplices that collapsed into each simplified one.
        # We'll pick one representative for the visual trace but we have access to all.
        orig_simplices = []
        for s in gen.support_simplices:
            orig_simplices.extend(simplex_map.get(s, [s]))
            
        orig_edges = []
        for e in gen.support_edges:
            orig_edges.extend(simplex_map.get(e, [e]))
            
        lifted_gens.append(HomologyGenerator(
            dimension=dim,
            support_simplices=list(set(orig_simplices)),
            support_edges=list(set(orig_edges)),
            weight=gen.weight
        ))
    return HomologyBasisResult(dimension=dim, rank=len(lifted_gens), generators=lifted_gens)

h0 = lift_h_basis(h0_qm, 0)
h1 = lift_h_basis(h1_qm, 1)
h2 = lift_h_basis(h2_qm, 2)

lifted_traces = []
for tr in pi1_qm.traces:
    orig_path = []
    if tr.vertex_path:
        # For paths, we take the representative from new_to_old_map
        # which is now new_v -> [old_v1, old_v2, ...]. We pick the first one.
        curr_orig_node = simplex_map.get((tr.vertex_path[0],), [(tr.vertex_path[0],)])[0][0]
        orig_path.append(curr_orig_node)
        
        for i in range(len(tr.vertex_path) - 1):
            u, v = tr.vertex_path[i], tr.vertex_path[i+1]
            simp_edge = tuple(sorted((u, v)))
            
            # Get an original edge that represents this simplified transition
            orig_edge_list = simplex_map.get(simp_edge, [(u, v)])
            orig_edge = orig_edge_list[0]
            
            target_u, target_v = orig_edge
            
            # Find shortest path from current position to either endpoint
            try:
                p_u = nx.shortest_path(orig_G, curr_orig_node, target_u)
                p_v = nx.shortest_path(orig_G, curr_orig_node, target_v)
                
                if len(p_u) <= len(p_v):
                    orig_path.extend(p_u[1:])
                    orig_path.append(target_v)
                    curr_orig_node = target_v
                else:
                    orig_path.extend(p_v[1:])
                    orig_path.append(target_u)
                    curr_orig_node = target_u
            except nx.NetworkXNoPath:
                # Fallback to direct representatives if path finding fails (should not happen in same component)
                next_v = simplex_map.get((v,), [(v,)])[0][0]
                orig_path.append(next_v)
                curr_orig_node = next_v
                
    lifted_traces.append(Pi1GeneratorTrace(
        generator=tr.generator,
        vertex_path=orig_path,
        component_root=simplex_map.get((tr.component_root,), [(tr.component_root,)])[0][0]
    ))
pi1 = Pi1PresentationWithTraces(generators=pi1_qm.generators, relations=pi1_qm.relations, traces=lifted_traces)

print("Invariants lifted.")

## Visualizing the Workspace

In [ ]:
res_html = visualize_topoview(
    sc, 
    dimension=3, 
    points=points_7d_torus,
    h0_basis=h0,
    h1_basis=h1,
    h2_basis=h2,
    pi1_result=pi1,
    title="7D Torus Workspace (Reduced)",
    features=["curvature", "dual", "components"],
    output_mode="full",
    show=False
)

with open("topoview_workspace.html", "w", encoding="utf-8") as f:
    f.write(res_html)

print("Workspace saved to topoview_workspace.html")